# v10 B300 — FlashInfer trtllm-gen NVFP4 comparator

The production bar: FlashInfer's `trtllm-gen` ships **NVFP4 KV decode** (released **v0.6.13**). Run it on
the **same shapes**, clock-locked, vs our `v10_nvfp4`. **Frame: complementing, not beating** (timestamped
June 2026) — the value is showing *where* the open kernel sits and *why* (the roofline read explains the
gap). **Two hard constraints from the API:** (1) the NVFP4-KV path requires an **FP8 query + FP8/NVFP4
output** — there is no fp16-Q + nvfp4-KV path, so note the precision asymmetry vs our FP16-Q kernel; (2)
**head_dim=128 only** (d=64 is not in the NVFP4-KV test matrix). Our pool `[num_blocks, page_size, H_kv,
d]` is exactly `kv_layout='NHD'` — no transpose. Install + dev-rung smoke this on the **B200** first.

## 0. Dependencies + GPU

In [1]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit('No GPU. FIX: rent a B200 (sm_100) or B300 (sm_103) on vast.ai — privileged/bare-metal for ncu.')

# matplotlib for the decisive plot (the one new dep vs other gates); numpy before torch.
pip('ninja', 'pytest', 'numpy', 'matplotlib')

try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False
if not cuda_ok:
    # Blackwell (B200 sm_100 / B300 sm_103) has no SASS in the cu124 wheel; nightly cu129 covers both.
    # (Skipped entirely if your image already ships a Blackwell-capable torch.)
    pip('--pre', 'torch', extra=('--index-url', 'https://download.pytorch.org/whl/nightly/cu129'))
    raise SystemExit('Installed Blackwell torch (nightly cu129). RESTART the kernel + re-run from the top.')

os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap,clocks.current.sm,clocks.max.sm --format=csv

torch 2.9.1+cu128 | cuda 12.8 | cap (10, 3)
name, compute_cap, clocks.current.sm [MHz], clocks.max.sm [MHz]
NVIDIA B300 SXM6 AC, 10.3, 2032 MHz, 2032 MHz
NVIDIA B300 SXM6 AC, 10.3, 120 MHz, 2032 MHz
NVIDIA B300 SXM6 AC, 10.3, 120 MHz, 2032 MHz
NVIDIA B300 SXM6 AC, 10.3, 120 MHz, 2032 MHz


## 1. Get the repo

In [2]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

Updating 125981f..e5ffc1f
Fast-forward
 notebooks/v10_b300_comparators.ipynb        |  57 +------
 notebooks/v10_b300_comparators_output.ipynb | 255 ++++++++++++++++++++++++++++
 2 files changed, 262 insertions(+), 50 deletions(-)
 create mode 100644 notebooks/v10_b300_comparators_output.ipynb
cwd /flashattention-cuda


From https://github.com/gkienpham-cmd/flashattention-cuda
 * branch            main       -> FETCH_HEAD
   125981f..e5ffc1f  main       -> origin/main


## 2. Install FlashInfer (released NVFP4 KV decode; prebuilt cubin avoids a long JIT)

In [3]:
import sys, subprocess
def pipi(*a): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a], check=False)
# v0.6.13 has the released NVFP4-KV decode path. cubin + jit-cache (cu129, covers 10.3a=B300) skip the
# first-call JIT compile. Match the index (cu128/cu129/cu130) to the box toolkit.
pipi('flashinfer-python==0.6.13')
pipi('flashinfer-cubin==0.6.13')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'flashinfer-jit-cache==0.6.13',
                '--index-url', 'https://flashinfer.ai/whl/cu129'], check=False)
import flashinfer; print('flashinfer', flashinfer.__version__)
# sanity: the decode entry + the NVFP4 quantizer must import
from flashinfer.decode import trtllm_batch_decode_with_kv_cache
from flashinfer.fp4_quantization import nvfp4_quantize_paged_kv_cache
print('NVFP4 decode API present OK')

flashinfer 0.6.13
NVFP4 decode API present OK


## 3. FlashInfer NVFP4 decode timing (B200 dev-rung smoke, then B300 record)

In [4]:
import torch, flashinfer
from flashinfer.fp4_quantization import nvfp4_quantize_paged_kv_cache

dev = 'cuda'
# head_dim=128 ONLY (the NVFP4-KV decode test matrix excludes d=64); G = H_q//H_kv.
B, H_q, H_kv, d, page_size = 64, 32, 8, 128, 16
seq_len = 8192
kv_layout = 'NHD'                       # matches our [num_blocks, page_size, H_kv, d] pool

pages_per_seq = (seq_len + page_size - 1) // page_size
num_pages = pages_per_seq * B
block_tables = torch.arange(num_pages, dtype=torch.int32, device=dev).reshape(B, pages_per_seq)
seq_lens = torch.full((B,), seq_len, dtype=torch.int32, device=dev)

k_bf16 = torch.randn(num_pages, page_size, H_kv, d, dtype=torch.bfloat16, device=dev)
v_bf16 = torch.randn(num_pages, page_size, H_kv, d, dtype=torch.bfloat16, device=dev)
(k_fp4, v_fp4), (k_sf, v_sf), k_scale, v_scale = nvfp4_quantize_paged_kv_cache(
    k_bf16, v_bf16, kv_layout=kv_layout)   # use the helper (V scales are swizzled — do NOT hand-roll)

# NVFP4 KV path REQUIRES an FP8 query + FP8/NVFP4 output (no fp16-Q path) — note the precision
# asymmetry vs our FP16-Q v10 kernel when reading the A/B.
q = torch.randn(B, H_q, d, dtype=torch.bfloat16, device=dev)
q_scale = (q.abs().amax() / 448.0).item()
q_fp8 = (q / q_scale).to(torch.float8_e4m3fn)
sm_scale = 1.0 / (d ** 0.5)
bmm1_scale = q_scale * k_scale * sm_scale
bmm2_scale = v_scale
workspace = torch.empty(256 * 1024 * 1024, dtype=torch.int8, device=dev)

# out_dtype must be a torch.dtype (not the string 'fp8'); the trtllm-gen FP8-output path accepts
# query.dtype (== float8_e4m3fn here), torch.float16 or torch.bfloat16. 'nvfp4' would also be valid
# but needs an o_sf_scale + uint8 FP4 output buffer — fp8 output is the apples-to-apples latency probe.
out_dtype = torch.float8_e4m3fn

def run_flashinfer():
    return flashinfer.decode.trtllm_batch_decode_with_kv_cache(
        q_fp8, (k_fp4, v_fp4), workspace, block_tables, seq_lens, int(seq_len),
        bmm1_scale, bmm2_scale, kv_layout=kv_layout, backend='trtllm-gen',
        out_dtype=out_dtype, kv_cache_sf=(k_sf, v_sf), q_len_per_req=1,
        uses_shared_paged_kv_idx=True)

for _ in range(10): run_flashinfer()
torch.cuda.synchronize()
s, e = torch.cuda.Event(True), torch.cuda.Event(True); s.record()
for _ in range(100): o = run_flashinfer()
e.record(); torch.cuda.synchronize()
print(f'FlashInfer trtllm-gen NVFP4 decode: {s.elapsed_time(e)/100*1e3:.2f} us/step  (B={B} Nk={seq_len} d={d} G={H_q//H_kv})')


[WARNING] NVFP4 KV cache with NHD layout will be converted to HND, incurring extra transpose and contiguous copy overhead. Use kv_layout='HND' for better performance.
[WARNING] NVFP4 KV cache with NHD layout will be converted to HND, incurring extra transpose and contiguous copy overhead. Use kv_layout='HND' for better performance.
[WARNING] NVFP4 KV cache with NHD layout will be converted to HND, incurring extra transpose and contiguous copy overhead. Use kv_layout='HND' for better performance.
[WARNING] NVFP4 KV cache with NHD layout will be converted to HND, incurring extra transpose and contiguous copy overhead. Use kv_layout='HND' for better performance.
[WARNING] NVFP4 KV cache with NHD layout will be converted to HND, incurring extra transpose and contiguous copy overhead. Use kv_layout='HND' for better performance.
[WARNING] NVFP4 KV cache with NHD layout will be converted to HND, incurring extra transpose and contiguous copy overhead. Use kv_layout='HND' for better performance

## 4. Our v10_nvfp4 on the SAME shapes (the A/B; note FP16-Q vs FlashInfer's FP8-Q)

In [5]:
# Time our kernel at the same B / N_k / d=128 / G via the regime helper, clock-matched in this process.
from bench.regime import _build_ours
import torch
dev='cuda'
B, H_kv, d, page_size, seq_len, G = 64, 8, 128, 256, 8192, 4
H_q = G * H_kv
q = torch.randn(B, H_q, 1, d, device=dev, dtype=torch.float16)
k = torch.randn(B, H_kv, seq_len, d, device=dev, dtype=torch.float16)
v = torch.randn(B, H_kv, seq_len, d, device=dev, dtype=torch.float16)
ours = _build_ours('v10_nvfp4', q, k, v, page_size, q_off=0)
for _ in range(10): ours()
torch.cuda.synchronize()
s, e = torch.cuda.Event(True), torch.cuda.Event(True); s.record()
for _ in range(100): ours()
e.record(); torch.cuda.synchronize()
print(f'v10_nvfp4 (ours, FP16-Q): {s.elapsed_time(e)/100*1e3:.2f} us/step  (B={B} Nk={seq_len} d={d} G={G})')
print('\nCompare to FlashInfer §3. Production-tuned trtllm-gen will likely win raw us/tok; the deliverable')
print('is WHERE the open kernel sits + WHY (roofline read), NOT a headline speedup. Mind the Q-precision gap.')

ninja: warning: build log version is too old; starting over
[1/3] c++ -MMD -MF binding.o.d -DTORCH_EXTENSION_NAME=fa_v10_nvfp4 -DTORCH_API_INCLUDE_EXTENSION_H -I/venv/main/lib/python3.12/site-packages/nvidia/cu13/include -I/venv/main/lib/python3.12/site-packages/nvidia/cublas/include -I/venv/main/lib/python3.12/site-packages/nvidia/cuda_cupti/include -I/venv/main/lib/python3.12/site-packages/nvidia/cuda_nvrtc/include -I/venv/main/lib/python3.12/site-packages/nvidia/cuda_runtime/include -I/venv/main/lib/python3.12/site-packages/nvidia/cudnn/include -I/venv/main/lib/python3.12/site-packages/nvidia/cufft/include -I/venv/main/lib/python3.12/site-packages/nvidia/cufile/include -I/venv/main/lib/python3.12/site-packages/nvidia/curand/include -I/venv/main/lib/python3.12/site-packages/nvidia/cusolver/include -I/venv/main/lib/python3.12/site-packages/nvidia/cusparse/include -I/venv/main/lib/python3.12/site-packages/nvidia/cusparselt/include -I/venv/main/lib/python3.12/site-packages/nvidia/nccl/i